# FDA Data Enrichment Pipeline

Validate ingredients with Groq API and enrich with FDA medical label data.

## Setup & Configuration

In [ ]:
import json
import os
import re
import csv
import time
import requests
import pandas as pd
from datetime import datetime

# Google Colab setup (uncomment if needed)
# from google.colab import drive
# drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/DataDoseDepi'

# Input: CSV output from active ingredient cleaning
CLEANED_CSV = os.path.join(BASE_DIR, 'DataDoseDataset_ActiveIngredient_Cleaned.csv')

# Outputs
OUTPUT_CSV = os.path.join(BASE_DIR, 'ingredients_fda_results.csv')
OUTPUT_JSON = os.path.join(BASE_DIR, 'ingredients_fda_results.json')
PROGRESS_FILE = os.path.join(BASE_DIR, 'fda_pipeline_progress.json')
LOG_FILE = os.path.join(BASE_DIR, 'fda_pipeline_log.txt')

# API Configuration
GROQ_API_KEYS = []  # Add your Groq API keys
GROQ_MODEL = "llama-3.1-8b-instant"
GROQ_URL = "https://api.groq.com/openai/v1/chat/completions"

OPENFDA_BASE = "https://api.fda.gov/drug/label.json"
OPENFDA_API_KEY = ""  # Optional OpenFDA key

print("Configuration loaded.")

## Step 1: Extract Unique Ingredients

In [ ]:
def extract_unique_ingredients(cleaned_csv):
    print(f"Reading: {cleaned_csv}")
    df = pd.read_csv(cleaned_csv)
    
    col = None
    for candidate in ['activeingredient_clean', 'Graph_Node_Ingredient', 'activeingredient']:
        if candidate in df.columns:
            col = candidate
            break
    
    if col is None:
        raise ValueError(f"Ingredient column not found. Available: {list(df.columns)}")
    
    print(f"Ingredient column: '{col}' | Total rows: {len(df):,}")
    
    all_ingredients = set()
    for row in df[col].dropna():
        parts = [p.strip() for p in str(row).split('+')]
        for p in parts:
            p = p.strip().lower()
            if p and len(p) > 2:
                all_ingredients.add(p)
    
    unique = sorted(all_ingredients)
    print(f"Unique ingredients: {len(unique):,}")
    return unique

unique_ingredients = extract_unique_ingredients(CLEANED_CSV)

## Step 2: Groq API Validation

In [ ]:
GROQ_SYSTEM_PROMPT = """You are a senior pharmaceutical scientist and medical terminologist.
Your job is to validate and canonicalize active pharmaceutical ingredient names.

INPUT: A drug ingredient name (possibly misspelled, abbreviated, or non-pharmaceutical).

YOUR TASK:
1. Is this a real, recognized ACTIVE PHARMACEUTICAL INGREDIENT (API)?
   - Yes: prescription drugs, OTC drugs, biologics, vaccines, vitamins (if pharmaceutical grade)
   - No: cosmetics, food ingredients, vague marketing terms

2. If YES — what is the CANONICAL INN (International Nonproprietary Name)?
   - Use WHO INN standard
   - Fix typos, expand abbreviations
   - Example: "paracetamol" → "paracetamol" (INN), NOT "acetaminophen"

3. REJECTION CRITERIA — return is_drug: false if:
   - It's a cosmetic ingredient
   - It's a vague term (minerals, vitamins, supplements)
   - It's a food/spice
   - Cannot be found in any drug regulatory database

RETURN FORMAT — ONLY valid JSON:
{"input": "input name", "is_drug": true/false, "canonical_name": "INN or null", 
"fda_search_term": "search term or null", "rejection_reason": "reason or null", "confidence": 0.0-1.0}"""

def call_groq_api(ingredient, api_key=None):
    if not api_key:
        return None
    
    headers = {
        'Authorization': f'Bearer {api_key}',
        'Content-Type': 'application/json',
    }
    
    payload = {
        'model': GROQ_MODEL,
        'messages': [
            {'role': 'system', 'content': GROQ_SYSTEM_PROMPT},
            {'role': 'user', 'content': f'Ingredient: {ingredient}'}
        ],
        'temperature': 0.1,
        'max_tokens': 200,
    }
    
    try:
        response = requests.post(GROQ_URL, json=payload, headers=headers, timeout=10)
        if response.status_code == 200:
            data = response.json()
            content = data['choices'][0]['message']['content']
            return json.loads(content)
    except Exception as e:
        pass
    
    return None

print("Groq API function ready.")

## Step 3: OpenFDA API Lookup

In [ ]:
def query_openfda(ingredient):
    search_term = ingredient.lower()
    params = {
        'search': f'active_ingredients.name:"{search_term}"',
        'limit': 1,
    }
    
    if OPENFDA_API_KEY:
        params['api_key'] = OPENFDA_API_KEY
    
    try:
        response = requests.get(OPENFDA_BASE, params=params, timeout=10)
        if response.status_code == 200:
            data = response.json()
            if data.get('results'):
                result = data['results'][0]
                return {
                    'found': True,
                    'brand_names': result.get('brand_name', []),
                    'generic_names': result.get('generic_name', []),
                    'manufacturers': result.get('manufacturer_name', []),
                    'dosage_forms': result.get('dosage_form', []),
                    'warnings': result.get('warnings', []),
                }
    except Exception as e:
        pass
    
    return {'found': False}

print("OpenFDA API function ready.")

## Step 4: Filter Confirmed Drugs

In [ ]:
def filter_confirmed_drugs(input_json, output_json=None, output_csv=None):
    print(f"Reading: {input_json}")
    
    with open(input_json, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    total = len(data)
    print(f"Total ingredients: {total:,}")
    
    confirmed = {}
    for ingredient, entry in data.items():
        groq = entry.get('groq_validation') or {}
        fda = entry.get('fda_data') or {}
        
        if groq.get('is_drug') and fda.get('found'):
            confirmed[ingredient] = {
                'ingredient': ingredient,
                'is_drug': True,
                'canonical_name': groq.get('canonical_name'),
                'fda_found': True,
                'brand_names': fda.get('brand_names', []),
                'generic_names': fda.get('generic_names', []),
                'dosage_forms': fda.get('dosage_forms', []),
            }
    
    print(f"\nResults:")
    print(f"  Confirmed (is_drug=true + fda_found=true): {len(confirmed):,}")
    print(f"  Rejected: {total - len(confirmed):,}")
    
    if output_json:
        with open(output_json, 'w', encoding='utf-8') as f:
            json.dump(confirmed, f, indent=2, ensure_ascii=False)
        print(f"  Saved to: {output_json}")
    
    if output_csv:
        confirm_df = pd.DataFrame(list(confirmed.values()))
        confirm_df.to_csv(output_csv, index=False)
        print(f"  Saved to: {output_csv}")
    
    return confirmed

print("Filter function ready.")

## Step 5: Full Pipeline Execution

In [ ]:
def run_full_pipeline(sample_size=10):
    print("Starting FDA enrichment pipeline...")
    print(f"Sample size: {sample_size} ingredients")
    
    # Process sample of ingredients
    results = []
    for i, ingredient in enumerate(unique_ingredients[:sample_size]):
        groq_result = None
        fda_result = {'found': False}
        
        if GROQ_API_KEYS:
            groq_result = call_groq_api(ingredient, GROQ_API_KEYS[0])
            if groq_result and groq_result.get('is_drug'):
                canonical = groq_result.get('canonical_name', ingredient)
                fda_result = query_openfda(canonical)
        
        results.append({
            'ingredient': ingredient,
            'groq_is_drug': groq_result.get('is_drug', False) if groq_result else False,
            'groq_canonical': groq_result.get('canonical_name') if groq_result else None,
            'groq_confidence': groq_result.get('confidence', 0) if groq_result else 0,
            'fda_found': fda_result.get('found', False),
            'brand_names': ', '.join(fda_result.get('brand_names', [])),
            'dosage_forms': ', '.join(fda_result.get('dosage_forms', [])),
        })
        
        if (i + 1) % 5 == 0:
            print(f"  Processed: {i + 1} / {sample_size}")
    
    results_df = pd.DataFrame(results)
    results_df.to_csv(OUTPUT_CSV, index=False)
    print(f"\nResults saved to: {OUTPUT_CSV}")
    
    print("\nSample results:")
    print(results_df.to_string())
    
    return results_df

# Uncomment to run
# results = run_full_pipeline(sample_size=20)

## Summary

In [ ]:
print("FDA Enrichment Pipeline")
print("=" * 50)
print("\nThis pipeline:")
print("  1. Extracts unique cleaned ingredients")
print("  2. Validates with Groq API (canonicalization)")
print("  3. Queries OpenFDA for drug metadata")
print("  4. Filters confirmed drugs (is_drug + fda_found)")
print("  5. Exports results as CSV and JSON")
print("\nOutputs:")
print(f"  - {OUTPUT_CSV}")
print(f"  - {OUTPUT_JSON}")
print(f"\nNext: Run run_full_pipeline(sample_size=N)")